### 42028 : Deep Learning and CNN
### Assignment 1

| Name | Swapnil Ajit Chhatre |
| :--- | :--- |
| Student ID | 25675238 |
| Tutorial | 04 (Wednesday 12.00 - 15.00) |
| Tutor | Mr. Deep Patel |

#### Aim
- To develop SVM and Neural Network classifiers for handwritten mathematical symbol classification.
- To use Histogram Of Oriented Gradients (HoG), Local Binary Pattern (LBP), Raw images/pixels, and any other feature for classification

### Contents
1. Section 1 : Feature Extraction
2. Section 2 : Splitting Dataset

### File Outcome
*A set of files which contain training and testing data of extracted image features and corresponding labels*

### Library Imports

In [1]:
#Importing libraries required in this file
import numpy as np
from skimage import feature
from sklearn.model_selection import train_test_split

### Section 1 : Feature Extraction (HoG, LBP, GLCM, and Gabor)

*Feature extraction is an important in any computer vision task as it transforms raw pixel data into identifiable information such as edges, texture, shapes, etc. which is required for object detection, segmentation, recognition, and classification. Feature extraction also helps in reducing the dimensionality of the image which enables the algorithms to focus on important information instead of irrelevant parts of the image.*

*In this section, the following four feature extraction techniques are used to convert the processed images from previous section into useful features*

1. Histogram of Gradients (HoG) : Represents the distribution of local gradients in form of histograms of orientation.
2. Local Binary Patterns (LBP) : LBP captures local texture information by defining a binary pattern for each pixel by comparing the intensity values of its neighbours and using a threshold to converting these intensity values to 0 or 1

In [2]:
#Loading processed images from previous section
processed_images = np.load('./features/processed_images.npy')
labels = np.load('./original_data/labels.npy')

#### 1. Histogram of Gradients

In [4]:
#Extracting features using HoG
hog_features = []
for img in processed_images:
    hog = feature.hog(
        image=img,
        orientations=9,
        pixels_per_cell=(15,15),
        cells_per_block=(2,2)
    )
    hog_features.append(hog)
print("Total number of HoG Image Features :", len(hog_features))

Total number of HoG Image Features : 5000


In [5]:
print(len(hog_features[0]))

144


In [6]:
#Saving HoG features to a .npy file for reuse
hog_np_array = np.array(hog_features)
np.save('./features/hog_features.npy', hog_features)

##### HoG Feature Vector Length Calculation

*It can be observed that the length of the HoG feature is 144. The calculation can be explained as follows-*

**Number of Cells**

= (Image length / Vertical no. of pixels per cell) x (Image width / Horizontal no. of pixels per cell)

= (45/15) x (45/15)

= 3 x 3 cells in total

**Number of Blocks**

= (Number of horizontal cells - Number of horizontal cells per block + 1) x (Number of vertical cells - Number of vertical cells per block + 1)

= (3 - 2 + 1) x (3 - 2 + 1)

= 2 x 2 = 4 blocks

**Features per block**

= Orientation x Total Cells

= 9 bins x 4 blocks = 36 features per block

**Total features**

= Number of blocks x Features per block

= 4 x 36 = *144*

#### 2. Local Binary Pattern

In [7]:
#Function to calculate spatial histograms for cells

def calculate_histogram(lbp):
    cell_size = 15
    histograms = []
    for i in range(0, 45, cell_size):
        for j in range(0, 45, cell_size):
            cell = lbp[i : i + cell_size, j : j + cell_size]

            hist, _ = np.histogram(
                cell.ravel(),
                bins = np.arange(0, 11),
                range = (0, 10)
            )

            hist = np.float32(hist)
            hist /= (hist.sum() + 1e-7)
            histograms.append(hist)
    lbp_feature_vector = np.concatenate(histograms)
    return lbp_feature_vector

In [8]:
#Extracting features using HoG
lbp_features = []
P_value = 8 #Number of neighbours to be considered
R_value = 1 #Radius of circle = P/8

for img in processed_images:
    lbp = feature.local_binary_pattern(
        image=img,
        P=P_value,
        R=R_value,
        method="uniform",
    )

    lbp_feature_vector = calculate_histogram(lbp)
    lbp_features.append(lbp_feature_vector)

print("Total number of LBP Image Features :", len(lbp_features))

/opt/conda/lib/python3.12/site-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


Total number of LBP Image Features : 5000


In [9]:
print(len(lbp_features[0]))

90


In [11]:
#Saving LBP features to a .npy file for reuse
lbp_np_array = np.array(lbp_features)
np.save('./features/lbp_features.npy', lbp_features)

##### LBP Feature Vector Length Calculation

*It can be observed that the length of the LBP feature is 90. The calculation can be explained as follows-*

**1.  Divide the image into a 3x3 grid**

= Cell Size =  15X15 = 9 cells

**2. Iterative Histogram Calculation**

1. For each 15x15 LBP path
2. Calculate histogram function using P + 2 (10) bins

**3. Feature length calculation**

= No. of cells X No. of bins / cell = 9 X 10 = 90 features

### Section 2 : Splitting the Dataset

In [3]:
#Importing raw grayscale images, hog features, and lbp features

raw_pixel = np.load('./features/processed_images.npy')
hog_features = np.load('./features/hog_features.npy')
lbp_features = np.load('./features/lbp_features.npy')

In [4]:
#Flattening raw_pixel values
raw_pixel = raw_pixel.reshape(-1,2025)

In [5]:
print("Raw pixel feature length :" , raw_pixel.shape)
print("HOG feature length :" , hog_features.shape)
print("LBP feature length :" , lbp_features.shape)

Raw pixel feature length : (5000, 2025)
HOG feature length : (5000, 144)
LBP feature length : (5000, 90)


In [6]:
#Importing labels
labels = np.load('./original_data/labels.npy')

In [7]:
#Splitting raw image, HoG features and LBP features
#Using 70% data for training and 30% for testing

raw_pixel_x_train, raw_pixel_x_test, raw_pixel_y_train, raw_pixel_y_test = train_test_split(
    raw_pixel, labels,
    test_size=0.3,
    train_size=0.7,
    random_state=44
)
hog_x_train, hog_x_test, hog_y_train, hog_y_test = train_test_split(
    hog_features, labels,
    test_size=0.3,
    train_size=0.7,
    random_state=44
)
lbp_x_train, lbp_x_test, lbp_y_train, lbp_y_test = train_test_split(
    lbp_features, labels,
    test_size=0.3,
    train_size=0.7,
    random_state=44
)

In [9]:
#Displaying train and test set sizes

print("Raw pixel training set images :", len(raw_pixel_x_train))
print("Raw pixel training set labels :", len(raw_pixel_y_train))
print("Raw pixel testing set images :", len(raw_pixel_x_test))
print("Raw pixel testing set labels :", len(raw_pixel_y_test))

print("HOG training set images :", len(hog_x_train))
print("HOG training set labels :", len(hog_y_train))
print("HOG testing set images :", len(hog_x_test))
print("HOG testing set labels :", len(hog_y_test))

print("LBP training set images :", len(lbp_x_train))
print("LBP training set labels :", len(lbp_y_train))
print("LBP testing set images :", len(lbp_x_test))
print("LBP testing set labels :", len(lbp_y_test))

Raw pixel training set images : 3500
Raw pixel training set labels : 3500
Raw pixel testing set images : 1500
Raw pixel testing set labels : 1500
HOG training set images : 3500
HOG training set labels : 3500
HOG testing set images : 1500
HOG testing set labels : 1500
LBP training set images : 3500
LBP training set labels : 3500
LBP testing set images : 1500
LBP testing set labels : 1500


In [11]:
#Saving training and testing data splits for raw pixels, HoG features, and LBP features

np.save('./train/raw_pixel_train_images.npy', raw_pixel_x_train)
np.save('./train/raw_pixel_train_labels.npy', raw_pixel_y_train)
np.save('./test/raw_pixel_test_images.npy', raw_pixel_x_test)
np.save('./test/raw_pixel_test_labels.npy', raw_pixel_y_test)

np.save('./train/hog_train_images.npy', hog_x_train)
np.save('./train/hog_train_labels.npy', hog_y_train)
np.save('./test/hog_test_images.npy', hog_x_test)
np.save('./test/hog_test_labels.npy', hog_y_test)

np.save('./train/lbp_train_images.npy', lbp_x_train)
np.save('./train/lbp_train_labels.npy', lbp_y_train)
np.save('./test/lbp_test_images.npy', lbp_x_test)
np.save('./test/lbp_test_labels.npy', lbp_y_test)